# Bootstrapped flow: train and predict

This notebook trains a new conditional redshift flow from `data/parent.npz`,
predicts on held-out broadband photometry, and saves/reloads the model.
It shows the training loop directly, using the architecture and numerical
helpers from `bin/train_flows.py`. No existing model checkpoint is required.

Use the project's Conda environment and the **Python (pzflow)** kernel.
Run from the checkout or `nb/`. Outputs remain local under `results/bootstrapped_flow_demo/`;
save executed notebooks under `results/notebooks_executed/`.
Defaults match the main pipeline: seed 20261024, 200 epochs maximum,
20,000 optimizer steps maximum, four CPU threads, and validation early stopping.
For a short trial, reduce `epochs` and `max_steps` in the first cell.

These demos reproduce the **noise-augmented main flow experiment**, not the
separate fixed-photometry flow/NoiseSOM control or the paper's frozen older runs.
Native CFHT u/HSC grizy filters are retained; LSST-like refers to the noise level.

## What is bootstrapped?

Each training galaxy supplies a full COSMOS multiband redshift PDF, not an exact
redshift. For each batch, draw z from that PDF and draw broadband photometric
noise independently conditional on the galaxy's deep observations. Minimizing
−log q(z | x) over repeated draws averages over the uncertain labels.

All photometric training galaxies are used, including those without spectroscopy.
Validation uses four cached PDF draws per galaxy, never spectroscopic labels.
No division by a redshift prior or recalibration of the input PDFs is performed.
The learned distribution can inherit input-PDF biases and prior dependence.

## 1. Imports, paths, and training limits

The model has two conditional rational-quadratic spline transforms (16 bins,
64 hidden features, two residual blocks) and a standard-normal base distribution.
We reuse that exact definition rather than introduce a second model implementation.

In [ ]:
from pathlib import Path
import copy
import hashlib
import json
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import torch

root = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'bin/train_flows.py').exists()
)
sys.path.insert(0, str(root / 'bin'))
sys.path.insert(0, str(root / 'src'))
from train_flows import (
    context,
    log_prob_z,
    make_flow,
    metrics,
    posterior_summary,
    raw_features,
    t_to_z,
    z_to_t,
)

parent_path = root / 'data/parent.npz'
out_dir = root / 'results/bootstrapped_flow_demo'
name = 'bootstrapped'
seed = 20261024
epochs = 200
max_steps = 20000
threads = 4
if epochs < 1 or max_steps < 1:
    raise ValueError('epochs and max_steps must be positive')
if out_dir.exists() and any(out_dir.iterdir()):
    raise FileExistsError(f'Choose an empty output directory: {out_dir}')
torch.set_num_threads(threads)

plt.rcParams.update(
    {
        'font.family': 'serif',
        'font.size': 11,
        'axes.linewidth': 1.5,
        'xtick.major.size': 5,
        'ytick.major.size': 5,
        'xtick.major.width': 1.5,
        'ytick.major.width': 1.5,
        'legend.frameon': False,
    }
)

## 2. Load the parent and preserve the spatial split

Fluxes and errors are in microJy, ordered u, g, r, i, z, y, with shape (N, 6).
`native_flux` is the deep measurement; `sigma_added` is the extra noise standard
deviation needed for the target depth. The saved `flux` is one fixed degraded
realization used for validation and testing. We do not add noise to it again.
The split is spatial: 0 = training, 1 = validation, 2 = test.

In [ ]:
keys = [
    'id',
    'native_flux',
    'flux',
    'fluxerr',
    'sigma_added',
    'zspec',
    'has_specz',
    'split',
    'imag',
    'cell',
    'grid',
    'pdf',
    'z_draws',
]
with np.load(parent_path) as f:
    metadata = json.loads(str(f['metadata']))
    if not metadata.get('canonical_parent'):
        raise ValueError('Use the canonical data/parent.npz')
    data = {key: f[key] for key in keys}
with parent_path.open('rb') as f:
    parent_sha256 = hashlib.file_digest(f, 'sha256').hexdigest()

assert data['flux'].shape == data['fluxerr'].shape == (len(data['id']), 6)
assert np.isfinite(data['flux']).all() and np.isfinite(data['native_flux']).all()
assert np.isfinite(data['fluxerr']).all() and (data['fluxerr'] > 0).all()
assert np.isfinite(data['sigma_added']).all() and (data['sigma_added'] >= 0).all()
assert len(np.unique(data['id'])) == len(data['id'])
for a, b in [(0, 1), (0, 2), (1, 2)]:
    assert not np.intersect1d(
        data['cell'][data['split'] == a], data['cell'][data['split'] == b]
    ).size
spec = data['has_specz']
assert np.all((data['zspec'][spec] > 0) & (data['zspec'][spec] < 10))
print(f'Parent: {len(data["id"]):,} galaxies; SHA256 {parent_sha256}')

## 3. Normalize the broadband context using training photometry

Each context has 12 features: six asinh magnitude features (defined for negative
fluxes) and six log errors. Match `train_flows.py`: estimate means and scales
from the **full photometric training split**, without using any redshift labels.
Both flow demos use this same normalization, even though their labeled samples differ.

In [ ]:
phot_train = data['split'] == 0
features = raw_features(data['native_flux'][phot_train], data['fluxerr'][phot_train])
mean = features.mean(axis=0)
scale = np.maximum(features.std(axis=0), 0.1)
assert mean.shape == scale.shape == (12,)
del features
print('Context features: six transformed fluxes + six log errors')

## 4. Select training and validation labels

Select a fixed subset of up to 4,096 validation galaxies using the training seed.
Fixed validation photometry and labels let us compare losses across epochs.
The held-out test set never selects an epoch or a hyperparameter.

In [ ]:
from pzflow.utils import PDFSampler

assert data['grid'][0] == 0 and data['grid'][-1] == 10
sampler = PDFSampler(data['grid'], data.pop('pdf'))
train = np.flatnonzero(data['split'] == 0)
valid = np.flatnonzero(data['split'] == 1)
rng = np.random.default_rng(seed)
vi = rng.choice(valid, min(len(valid), 4096), replace=False)
vy = data['z_draws'][vi, :4].reshape(-1)
vi = np.repeat(vi, 4)
assert np.isfinite(vy).all() and np.all((vy >= 0) & (vy <= 10))
print(f'Training: {len(train):,}; validation: {len(valid):,} galaxies')
print(f'Fixed validation labels: {len(vy):,} (four draws per selected galaxy)')

## 5. Initialize the flow and optimizer

The flow models a scaled-logit redshift variable t on the real line, mapping
0 < z < 10 to the full support of its normal base. We maximize density in t
while training. The redshift Jacobian is independent of model parameters, so
omitting it during optimization changes no gradients; include it when evaluating
PDFs in physical redshift units with `log_prob_z`.

In [ ]:
assert len(train) and len(valid)
vx = context(data['flux'][vi], data['fluxerr'][vi], mean, scale)
vt = z_to_t(torch.from_numpy(vy[:, None]))

torch.manual_seed(seed)
model = make_flow()
opt = torch.optim.AdamW(model.parameters(), lr=8e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3, factor=0.5)
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters()):,}')

## 6. Train, validate, and retain the best epoch

The loop follows `_fit_flow` in `train_flows.py`. Every batch starts from native
fluxes and draws only the missing noise variance. It uses batches of 512,
gradient clipping at 5, learning-rate reduction after a validation plateau,
and early stopping after eight epochs without an improvement of at least 1e-4.
The validation loss below is a density in transformed t, not a redshift-space score.

In [ ]:
best = float('inf')
best_state = None
bad = 0
step = 0
history = []
start = time.monotonic()
for epoch in range(epochs):
    model.train()
    rng.shuffle(train)
    losses = []
    for pos in range(0, len(train), 512):
        idx = train[pos : pos + 512]
        noise = rng.normal(size=(len(idx), 6)).astype(np.float32)
        xflux = data['native_flux'][idx] + noise * data['sigma_added'][idx]
        x = context(xflux, data['fluxerr'][idx], mean, scale)
        z = sampler.draw(idx, rng)
        target = z_to_t(torch.from_numpy(z[:, None]))

        opt.zero_grad(set_to_none=True)
        loss = -model.log_prob(target, context=x).mean()
        if not torch.isfinite(loss):
            raise RuntimeError('Nonfinite training loss')
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        losses.append(float(loss.detach()))
        step += 1
        if step >= max_steps:
            break

    model.eval()
    with torch.no_grad():
        vals = [
            -model.log_prob(vt[p : p + 1024], context=vx[p : p + 1024]).sum().item()
            for p in range(0, len(vt), 1024)
        ]
        val = sum(vals) / len(vt)
    scheduler.step(val)
    history.append(
        {
            'epoch': epoch + 1,
            'step': step,
            'train_loss': float(np.mean(losses)),
            'validation_loss': val,
        }
    )
    print(f'Epoch {epoch + 1}; step {step}; train {np.mean(losses):.4f}; val {val:.4f}')
    if val < best - 1e-4:
        best = val
        best_state = copy.deepcopy(model.state_dict())
        bad = 0
    else:
        bad += 1
    if bad >= 8 or step >= max_steps:
        break

assert best_state is not None
model.load_state_dict(best_state)
model.eval()
training_seconds = time.monotonic() - start
print(f'Best validation loss: {best:.4f}; training time: {training_seconds:.1f} s')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5), constrained_layout=True)
ax.plot([row['epoch'] for row in history], [row['train_loss'] for row in history], label='Train')
ax.plot(
    [row['epoch'] for row in history],
    [row['validation_loss'] for row in history],
    label='Validation',
)
ax.set(xlabel='Epoch', ylabel='Negative log density in transformed redshift')
ax.legend()
plt.show()

## 7. Predict from broadband fluxes and errors alone

Inference does not require spectroscopy or COSMOS PDFs. The function below
transforms each batch's fluxes/errors into the saved context and inverts five
standard-normal quantiles through the flow, then maps t back to z. The columns
are the 2.5%, 16%, 50%, 84%, and 97.5% redshift quantiles.

Apply it to the **full photometric test split**, including galaxies without
spectroscopy. We only restrict to secure spectra later when evaluating accuracy.

In [ ]:
@torch.no_grad()
def predict_quantiles(model, flux, err, mean, scale, batch_size=512):
    # Return five redshift quantiles for each six-band flux/error row.
    probs = torch.tensor([0.025, 0.16, 0.5, 0.84, 0.975])
    normal_q = torch.distributions.Normal(0.0, 1.0).icdf(probs)
    prediction = []
    for start in range(0, len(flux), batch_size):
        x = context(flux[start : start + batch_size], err[start : start + batch_size], mean, scale)
        t, _ = model._transform.inverse(
            normal_q.repeat(len(x)).reshape(-1, 1),
            context=x.repeat_interleave(5, dim=0),
        )
        prediction.append(t_to_z(t).reshape(len(x), 5).numpy())
    return np.concatenate(prediction)


phot_test = np.flatnonzero(data['split'] == 2)
q_all = predict_quantiles(model, data['flux'][phot_test], data['fluxerr'][phot_test], mean, scale)
assert q_all.shape == (len(phot_test), 5)
assert np.isfinite(q_all).all() and np.all(np.diff(q_all, axis=1) >= 0)
assert np.all((q_all >= 0) & (q_all <= 10))
print(f'Predicted {len(phot_test):,} test galaxies without using redshift labels')

## 8. Evaluate only where independent test spectroscopy is available

Now use the secure test spectra to measure normalized bias/scatter, outliers,
PIT, negative log density, and central-interval coverage. The numerical helpers
are the same ones used by the main script. Test labels do not affect training
or the predictions above. These metrics describe a spectroscopically selected
subset, not calibration of the full faint photometric population.

In [ ]:
scored = data['has_specz'][phot_test]
test = phot_test[scored]
ztest = data['zspec'][test]
q = q_all[scored]
ctx = context(data['flux'][test], data['fluxerr'][test], mean, scale)
q_check, pit, nll = posterior_summary(model, ctx, ztest)
np.testing.assert_allclose(q, q_check, atol=2e-6, rtol=1e-5)
result = metrics(ztest, q, pit, nll)
print(json.dumps(result, indent=2))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), constrained_layout=True)
axes[0].hexbin(ztest, q[:, 2], gridsize=60, bins='log', mincnt=1, cmap='viridis')
axes[0].plot([0, 5], [0, 5], 'k--', lw=1)
axes[0].set(xlim=(0, 5), ylim=(0, 5), xlabel=r'$z_{\mathrm{spec}}$', ylabel=r'Median $z$')
credibility = np.linspace(0, 1, 101)
rank = np.abs(2 * pit - 1)
curve = np.array([(rank <= level).mean() for level in credibility])
curve[0], curve[-1] = 0, 1
axes[1].plot(credibility, curve, label='Flow')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Nominal')
axes[1].set(xlim=(0, 1), ylim=(0, 1), xlabel='Nominal credibility', ylabel='Test coverage')
axes[1].legend()
plt.show()

Example PDFs below use `log_prob_z`, including the redshift Jacobian.
Examples span the spectroscopic test sample's native i-band magnitudes.
Dashed lines show test spectroscopy for evaluation only.

In [ ]:
order = np.argsort(data['imag'][test])
examples = order[np.linspace(0, len(order) - 1, 6, dtype=int)]
zgrid = np.linspace(1e-4, 10 - 1e-4, 5001, dtype=np.float32)
fig, axes = plt.subplots(2, 3, figsize=(10, 5.5), constrained_layout=True)
for ax, j in zip(axes.flat, examples):
    x = ctx[j : j + 1].repeat(len(zgrid), 1)
    pdf = np.exp(log_prob_z(model, zgrid, x).numpy())
    assert np.isfinite(pdf).all() and (pdf >= 0).all()
    ax.plot(zgrid, pdf, label='Flow PDF')
    ax.axvline(ztest[j], color='k', ls='--', label=r'$z_{\mathrm{spec}}$')
    ax.set(xlim=(0, min(10, max(q[j, 4], ztest[j]) + 0.3)), xlabel=r'$z$', ylabel=r'$p(z)$')
    ax.set_title(f'ID {data["id"][test[j]]}; i = {data["imag"][test[j]]:.2f}')
axes.flat[0].legend(fontsize=8)
plt.show()

## 9. Save the checkpoint, predictions, and provenance

The checkpoint uses the same fields as `train_flows.py`, including the feature
normalization needed at inference. Store full photometric test quantiles
separately from the scored spectroscopic test predictions. Existing scientific
runs are untouched. Training/source hashes and limits are recorded in `run.json`.

In [ ]:
out_dir.mkdir(parents=True, exist_ok=True)
checkpoint = {
    'state_dict': best_state,
    'feature_mean': mean,
    'feature_scale': scale,
    'model': 'two conditional RQS autoregressive transforms; 16 bins; hidden64; blocks2',
    'n_train': len(train),
    'n_validation': len(valid),
    'seed': seed,
}
torch.save(checkpoint, out_dir / f'{name}.pt')
with (out_dir / f'{name}_history.json').open('w') as f:
    json.dump(history, f, indent=2)
    f.write(chr(10))
np.savez_compressed(
    out_dir / 'photometric_test_predictions.npz',
    id=data['id'][phot_test],
    flux=data['flux'][phot_test],
    fluxerr=data['fluxerr'][phot_test],
    quantiles=q_all,
)
prediction = {key: data[key][test] for key in ['id', 'zspec', 'imag', 'flux', 'fluxerr']}
prediction.update({f'{name}_quantiles': q, f'{name}_pit': pit, f'{name}_nll': nll})
np.savez_compressed(out_dir / 'predictions.npz', **prediction)
run = {
    'model': name,
    'seed': seed,
    'threads': threads,
    'parent_sha256': parent_sha256,
    'epochs_limit': epochs,
    'steps_limit': max_steps,
    'epochs': len(history),
    'steps': step,
    'n_train': len(train),
    'n_validation': len(valid),
    'n_photometric_test': len(phot_test),
    'n_spec_test': len(test),
    'best_validation_loss': best,
    'training_seconds': training_seconds,
    'photometry': 'native CFHT/HSC with fresh missing-variance noise during training',
    'feature_normalization': 'full native photometric training split; no redshift labels',
    'metrics': result,
    'source_sha256': {
        path: hashlib.sha256((root / path).read_bytes()).hexdigest()
        for path in ['bin/train_flows.py', 'src/pzflow/utils.py']
    },
    'versions': {'numpy': np.__version__, 'torch': torch.__version__},
}
with (out_dir / 'run.json').open('w') as f:
    json.dump(run, f, indent=2)
    f.write(chr(10))
print(f'Saved model and predictions to {out_dir}')

## 10. Reload and apply to new broadband measurements

Recreate the architecture, restore the weights and preprocessing, and predict.
`new_flux` and `new_err` can be replaced with any (N, 6) arrays in the same units
and band order. They must be finite and errors positive; redshift labels are
not needed. Use `torch.load` only for a checkpoint you trust.

In [ ]:
saved = torch.load(out_dir / f'{name}.pt', map_location='cpu', weights_only=False)
reloaded = make_flow()
reloaded.load_state_dict(saved['state_dict'])
reloaded.eval()
new_flux = data['flux'][phot_test[:5]]
new_err = data['fluxerr'][phot_test[:5]]
assert np.isfinite(new_flux).all() and np.isfinite(new_err).all() and (new_err > 0).all()
new_q = predict_quantiles(
    reloaded, new_flux, new_err, saved['feature_mean'], saved['feature_scale']
)
np.testing.assert_allclose(new_q, q_all[:5], atol=2e-6, rtol=1e-5)
print('Columns: 2.5%, 16%, 50%, 84%, 97.5% redshift quantiles')
print(new_q)